In [8]:
import pandas as pd
import numpy as np
import pickle

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import mean_absolute_percentage_error

Inlezen data

In [9]:
kijkcijfers = pd.read_csv('./data2/feat_eng/kijkcijfers_weerdata.csv')
# SE = sentence embedding
kijkcijfers_SE = pd.read_csv('./data2/feat_eng/kijkcijfers_weerdata_met_sentence_embedding.csv')

# Opsplitsen in train en test set

In [10]:
cutoff = pd.Timestamp('2023-01-01')

kijkcijfers['timestamp'] = pd.to_datetime(kijkcijfers['timestamp'])

train_set_temp = kijkcijfers[kijkcijfers['timestamp'] < cutoff]
test_set_temp = kijkcijfers[kijkcijfers['timestamp'] >= cutoff]

train_set = train_set_temp.select_dtypes(include=[np.number])
test_set = test_set_temp.select_dtypes(include=[np.number])

X_train = train_set.drop(columns=['viewers'])
y_train = train_set['viewers']

X_test = test_set.drop(columns=['viewers'])
y_test = test_set['viewers']

In [11]:
cutoff = pd.Timestamp('2023-01-01')

kijkcijfers_SE['timestamp'] = pd.to_datetime(kijkcijfers_SE['timestamp'])

train_set_temp_SE = kijkcijfers_SE[kijkcijfers_SE['timestamp'] < cutoff]
test_set_temp_SE = kijkcijfers_SE[kijkcijfers_SE['timestamp'] >= cutoff]

train_set_SE = train_set_temp_SE.select_dtypes(include=[np.number])
test_set_SE = test_set_temp_SE.select_dtypes(include=[np.number])

X_train_SE = train_set_SE.drop(columns=['viewers'])
y_train_SE = train_set_SE['viewers']

X_test_SE = test_set_SE.drop(columns=['viewers'])
y_test_SE = test_set_SE['viewers']

Hulpfuncties om modellen te testen

In [12]:
def test_model(model, X, y, n_splits=5, scoring='neg_mean_absolute_error'):
    # Pipeline die voor trainen standard scaling toepast
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    # Om in plaats van gewone folds, gebruik te maken van TimeSeriesSplit om trends in de tijd te behouden
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    # Voer cross-validatie uit
    scores = cross_val_score(pipeline, X, y, cv=tscv, scoring=scoring, n_jobs=-1)
    
    # Gemiddelde en standaarddeviatie van de scores
    mean_score = -np.mean(scores)
    std_score = np.std(scores)
    
    print(f"Mean MAE: {mean_score:.3f} (+/- {std_score:.3f})")
    return scores

def test_model_list(list, X, y, n_splits=5, scoring='neg_mean_absolute_error'):
    for model in list:
        print(f"Testing model: {model.__class__.__name__}")
        test_model(model, X, y, n_splits=n_splits, scoring=scoring)

# Baseline models testing

In [13]:
ridge_reg = Ridge()

In [14]:
scores = test_model(ridge_reg, X_train, y_train)
scores_SE = test_model(ridge_reg, X_train_SE, y_train_SE)

print(f'Scores: {scores}\nScores_SE: {scores_SE}')

Mean MAE: 92790.431 (+/- 9726.184)


KeyboardInterrupt: 

# Other models testing

In [ ]:
rf_reg = RandomForestRegressor()
gb_reg = GradientBoostingRegressor()
xgb_reg = XGBRegressor()

In [ ]:
print("Without sentence embedding")
test_model_list([rf_reg, gb_reg, xgb_reg], X_train, y_train)

print("\nWith sentence embedding")
test_model_list([rf_reg, gb_reg, xgb_reg], X_train_SE, y_train_SE)

Without sentence embedding
Testing model: RandomForestRegressor
Mean MAE: 62890.985 (+/- 4959.426)
Testing model: GradientBoostingRegressor
Mean MAE: 69094.292 (+/- 5034.250)
Testing model: XGBRegressor
Mean MAE: 63209.376 (+/- 6686.836)

With sentence embedding
Testing model: RandomForestRegressor
Mean MAE: 62737.096 (+/- 4780.647)
Testing model: GradientBoostingRegressor
Mean MAE: 69110.550 (+/- 5595.626)
Testing model: XGBRegressor
Mean MAE: 65609.922 (+/- 7339.826)


Aan de hand van deze resultaten zal ik verder een XGBRegressor model proberen finetunen op de data zonder sentence embedding, omdat RandomForestRegressor extreem veel langer duurt om te trainen

In [ ]:
y_train.describe()

count    4.508700e+04
mean     4.555592e+05
std      2.831226e+05
min      1.588700e+04
25%      2.367115e+05
50%      3.682120e+05
75%      6.156235e+05
max      2.494114e+06
Name: viewers, dtype: float64

# Finetuning

In [15]:
from scipy.stats import uniform, randint

param_distributions = {
    'xgb__n_estimators': randint(100, 500),
    'xgb__learning_rate': uniform(0.01, 0.2),
    "xgb__min_child_weight": randint(1, 10),
    'xgb__max_depth': randint(3, 10),
    'xgb__subsample': uniform(0.6, 0.4),
    'xgb__colsample_bytree': uniform(0.6, 0.4),
    'xgb__gamma': uniform(0, 0.3)
}

tscv = TimeSeriesSplit(n_splits=5)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(n_jobs=-1))
])

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=500,  # Aantal combinaties om te testen
    scoring='neg_mean_absolute_error',
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

In [16]:
random_search.fit(X_train, y_train)

print("Best parameters found: ", random_search.best_params_)
print("Best score found: ", -random_search.best_score_)

tuned_model = random_search.best_estimator_

Fitting 5 folds for each of 500 candidates, totalling 2500 fits
Best parameters found:  {'xgb__colsample_bytree': np.float64(0.9669127652498348), 'xgb__gamma': np.float64(0.0905411267666526), 'xgb__learning_rate': np.float64(0.025959422249457166), 'xgb__max_depth': 9, 'xgb__min_child_weight': 5, 'xgb__n_estimators': 326, 'xgb__subsample': np.float64(0.6557567765776106)}
Best score found:  60055.5375


In [17]:
# Voorspellingen maken met het getunede model
y_pred = tuned_model.predict(X_test)

# MAPE berekenen
mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"MAPE: {mape:.4f}")

MAPE: 0.1882


Fitting 5 folds for each of 50 candidates, totalling 250 fits\
Best parameters found:  {\
    'xgb__colsample_bytree': np.float64(0.8493192507310232),\
    'xgb__gamma': np.float64(0.09926940745579475), \
    'xgb__learning_rate': np.float64(0.02271167005720473), \
    'xgb__max_depth': 9, \
    'xgb__min_child_weight': 8, \
    'xgb__n_estimators': 392, \
    'xgb__subsample': np.float64(0.8918424713352255)}\
Best score found:  60196.19921875\
\
MAPE: 0.1915

Fitting 5 folds for each of 500 candidates, totalling 2500 fits
Best parameters found:  {\
'xgb__colsample_bytree': np.float64(0.9504682673682698), \
'xgb__gamma': np.float64(0.026635697855998952), \
'xgb__learning_rate': np.float64(0.03387485492450079), \
'xgb__max_depth': 8, \
'xgb__min_child_weight': 4, \
'xgb__n_estimators': 310, \
'xgb__subsample': np.float64(0.7803206763176206)}\
Best score found:  59925.29453125\
\
MAPE: 0.1913

Fitting 5 folds for each of 500 candidates, totalling 2500 fits\
Best parameters found:  {'xgb__colsample_bytree': np.float64(0.9669127652498348), 'xgb__gamma': np.float64(0.0905411267666526), 'xgb__learning_rate': np.float64(0.025959422249457166), 'xgb__max_depth': 9, 'xgb__min_child_weight': 5, 'xgb__n_estimators': 326, 'xgb__subsample': np.float64(0.6557567765776106)}
Best score found:  60055.5375\
\
MAPE: 0.1882

In [19]:
with open('./models2/tuned_xgb_model_randomsearch.pkl', 'wb') as f:
    pickle.dump(tuned_model, f)